# Load the orders CSV

Download the configured HTTPS CSV and publish HOMEWORK.RAW.HOMEWORK_OBT. Run as HOMEWORK_ENGINEER with HOMEWORK_WH and the HOMEWORK_CSV_ACCESS integration enabled. See SETUP.md for configuration. Run all cells in order, then run the local dbt build.

Trial accounts disable external access by default. If integration creation is rejected, use the existing local loader.py URL route in SETUP.md. This notebook requires an account with external access enabled.

## 1. Imports

In [ ]:
import csv
import hashlib
import json
from pathlib import Path
import shutil
import tempfile
from urllib.parse import urlparse
from urllib.request import urlopen
import uuid


## 2. Setup

In [ ]:
CSV = "https://gist.githubusercontent.com/fm-dp/e5beb68cf717dc6a8db91250e9320b9e/raw/c442fe6feb1f73f9f9db04171aee6fe02505ceb8"

TARGET_TABLE = "HOMEWORK.RAW.HOMEWORK_OBT"

COLUMNS = (
    "customer_id", "customer_name", "customer_phone", "customer_email",
    "order_id", "order_date", "order_total", "order_items",
)

## 3. Download the CSV

In [ ]:
def fetch_csv(source: str, destination: Path) -> None:
    if urlparse(source).scheme != "https":
        raise ValueError("The source URL must use HTTPS")
    with urlopen(source, timeout=60) as response:
        if urlparse(response.geturl()).scheme != "https":
            raise ValueError("The source redirected away from HTTPS")
        with destination.open("wb") as handle:
            shutil.copyfileobj(response, handle)


temp_directory = tempfile.TemporaryDirectory(prefix="homework_")
csv_path = Path(temp_directory.name) / "homework.csv"
try:
    fetch_csv(CSV, csv_path)
except Exception:
    temp_directory.cleanup()
    raise
print(f"Downloaded {csv_path.stat().st_size:,} bytes")

## 4. Validate the source

In [ ]:
def validate_csv(path: Path) -> int:
    count = 0
    with path.open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.reader(handle, strict=True)
        if next(reader, None) != list(COLUMNS):
            raise ValueError("CSV header must match the eight homework columns in order")

        for row in reader:
            count += 1
            if len(row) != len(COLUMNS):
                raise ValueError(f"Record {count}: expected eight fields")
            try:
                items = json.loads(row[7])
            except json.JSONDecodeError as exc:
                raise ValueError(f"Record {count}: invalid order_items JSON") from exc
            if not isinstance(items, list) or not items:
                raise ValueError(f"Record {count}: order_items must be a nonempty array")

    if not count:
        raise ValueError("CSV contains no data rows")
    return count

try:
    source_rows = validate_csv(csv_path)
    source_sha256 = hashlib.sha256(csv_path.read_bytes()).hexdigest()
except Exception:
    temp_directory.cleanup()
    raise
print(f"Validated {source_rows:,} rows")
print(f"SHA-256: {source_sha256}")

## 5. Snowpark session

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

session.use_role("HOMEWORK_ENGINEER")
session.use_warehouse("HOMEWORK_WH")
session.use_database("HOMEWORK")
session.use_schema("RAW")

context = session.sql("""
    SELECT
        CURRENT_ROLE()      AS role,
        CURRENT_WAREHOUSE() AS warehouse,
        CURRENT_DATABASE()  AS database,
        CURRENT_SCHEMA()    AS schema
""").collect()[0]

print(context.as_dict())

## 6. Load function

In [ ]:
def last_query_id() -> str:
    return session.sql(
        "SELECT LAST_QUERY_ID()"
    ).collect()[0][0]


def load_snapshot(path: Path, expected_rows: int) -> dict:
    candidate_name = "LOAD_" + uuid.uuid4().hex.upper()
    candidate = f"HOMEWORK.RAW.{candidate_name}"
    stage = f"@%{candidate_name}"

    definition = ", ".join(
        f"{name} VARCHAR" for name in COLUMNS
    )

    session.sql(
        f"CREATE TEMPORARY TABLE {candidate} ({definition})"
    ).collect()

    try:
        put_results = session.file.put(
            local_file_name=str(path.resolve()),
            stage_location=stage,
            auto_compress=True,
            overwrite=True,
        )

        if not put_results:
            raise RuntimeError(
                "The CSV was not uploaded to the temporary stage"
            )

        copy_results = session.sql(
            f"""
            COPY INTO {candidate}
            FROM {stage}
            FILE_FORMAT = (
                TYPE = CSV
                SKIP_HEADER = 1
                FIELD_OPTIONALLY_ENCLOSED_BY = '"'
                ESCAPE_UNENCLOSED_FIELD = NONE
                EMPTY_FIELD_AS_NULL = FALSE
                NULL_IF = ()
                ERROR_ON_COLUMN_COUNT_MISMATCH = TRUE
                SKIP_BYTE_ORDER_MARK = TRUE
                ENCODING = 'UTF8'
            )
            ON_ERROR = ABORT_STATEMENT
            """
        ).collect()

        copy_query_id = last_query_id()
        copy_rows = [row.as_dict() for row in copy_results]

        def value(row: dict, name: str, default=None):
            return row.get(name, row.get(name.upper(), default))

        rows_loaded = sum(
            int(value(row, "rows_loaded", 0) or 0)
            for row in copy_rows
        )

        errors_seen = sum(
            int(value(row, "errors_seen", 0) or 0)
            for row in copy_rows
        )

        statuses = {
            value(row, "status")
            for row in copy_rows
        }

        if (
            not copy_rows
            or statuses != {"LOADED"}
            or errors_seen != 0
            or rows_loaded != expected_rows
        ):
            raise RuntimeError(
                "COPY did not load every source row; snapshot unchanged"
            )

        actual_rows = session.sql(
            f"SELECT COUNT(*) FROM {candidate}"
        ).collect()[0][0]

        if actual_rows != expected_rows:
            raise RuntimeError(
                "Source and candidate counts differ; snapshot unchanged"
            )

        session.sql(
            f"""
            CREATE TABLE IF NOT EXISTS {TARGET_TABLE}
            ({definition})
            """
        ).collect()

        session.sql(
            f"""
            INSERT OVERWRITE INTO {TARGET_TABLE}
            SELECT * FROM {candidate}
            """
        ).collect()

        publish_query_id = last_query_id()

        return {
            "rows_loaded": actual_rows,
            "copy_query_id": copy_query_id,
            "publish_query_id": publish_query_id,
        }
    finally:
        try:
            session.sql(f"DROP TABLE IF EXISTS {candidate}").collect()
        except Exception:
            print("End the notebook session to release the temporary table.")

## 7. Publish the snapshot

In [ ]:
try:
    load_result = load_snapshot(csv_path, source_rows)
finally:
    temp_directory.cleanup()
print(f"Published {load_result['rows_loaded']:,} rows to {TARGET_TABLE}.")

## 8. Load report

In [ ]:
report = {
    "source": CSV,
    "source_rows": source_rows,
    "sha256": source_sha256,
    **load_result,
}
print(json.dumps(report, indent=2))